# Using `Transitioner`

Examples of state inputs, transition strategies, estimator lifecycle, matrix configuration, labels, neuroimaging data, and outputs.

## Setup

The following small three-node network and three states are used throughout the examples.

In [ ]:
import numpy as np
from braincontrol.transitions import Transitioner

A = np.array([
    [0.0, 0.2, 0.1],
    [0.2, 0.0, 0.3],
    [0.1, 0.3, 0.0],
])

X = np.eye(3)

## State input

State input can be supplied in two ways:

1. A single state set `X`.
2. Separate initial and final state sets `X0` and `Xf`.

These representations cannot be mixed.

For two-dimensional tabular input, rows represent states and columns represent nodes. A one-dimensional input represents a single state.

### Single state set

In [ ]:
transitioner = Transitioner(
    T=1.0,
    transitions="directed",
)

energy = transitioner.fit_transform(A=A, X=X)
energy

The available transition strategies depend on the number of states in `X`:

| Strategy | One state | Multiple states | Meaning |
|---|---|---|---|
| `"directed"` | No | Yes | All directed transitions between distinct states |
| `"undirected"` | No | Yes | One transition for each pair of distinct states |
| `"directed_with_self"` | No | Yes | All directed transitions, including self-transitions |
| `"self"` | Yes | Yes | One self-transition for each state |

For a single state, `"self"` is the only valid strategy.

In [ ]:
single_state = np.array([1.0, 0.0, 0.0])

transitioner = Transitioner(T=1.0, transitions="self")
energy = transitioner.fit_transform(A=A, X=single_state)
energy

### Separate initial and final states

Use `X0` and `Xf` when initial and final states are represented by separate state sets.

In [ ]:
X0 = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
])

Xf = np.array([
    [0.0, 0.0, 1.0],
    [1.0, 0.0, 0.0],
])

transitioner = Transitioner(T=1.0, transitions="paired")
energy = transitioner.fit_transform(A=A, X0=X0, Xf=Xf)
energy

Two strategies are available:

- `"all_to_all"` computes every transition from a state in `X0` to a state in `Xf`.
- `"paired"` computes transitions between corresponding states in `X0` and `Xf`. The two inputs must contain the same number of states.

## Fit and transform

`fit` validates the estimator configuration and empirical inputs and learns the node schema:

In [ ]:
transitioner = Transitioner(T=1.0, transitions="directed")
transitioner.fit(A=A, X=X)

The fitted schema is stored in:

In [ ]:
transitioner.n_nodes_, transitioner.node_labels_

`transform` validates new inputs against that fitted schema and computes transition energy:

In [ ]:
energy = transitioner.transform(A=A, X=X)
energy

When fitting and transforming the same data, use:

In [ ]:
energy = transitioner.fit_transform(A=A, X=X)
energy

## Matrix inputs

`A` is the adjacency matrix and must have shape `(n_nodes, n_nodes)`.

The control input matrix `B` defaults to `"identity"`:

In [ ]:
energy = transitioner.fit_transform(A=A, B="identity", X=X)
energy

For optimal control energy, the state-trajectory constraint matrix `S` also defaults to `"identity"`.

All empirical matrix and state inputs must describe the same number of nodes.

## Adjacency matrix normalization

By default, `Transitioner` normalizes `A` during `transform` using `nctpy.utils.matrix_normalization` for the selected system.

In [ ]:
transitioner = Transitioner(T=1.0, system="continuous", c=2)

To use the adjacency matrix as supplied:

In [ ]:
transitioner = Transitioner(T=1.0, normalize_A=False)

## Energy type

Optimal control energy is the default. In the examples, `T` is specified as a float (e.g. `1.0`) for optimal energy.


In [ ]:
transitioner = Transitioner(T=1.0, energy_type="optimal", rho=1.0)

For optimal energy, `rho`, `S`, and `xr` participate in the control problem. `S` defaults to `"identity"` and `xr` defaults to `"xf"`.

Minimal control energy is selected explicitly:

In [ ]:
transitioner = Transitioner(T=1.0, energy_type="minimal", rho=None)

energy = transitioner.fit_transform(A=A, S=None, X=X, xr=None)
energy

For minimal energy, `rho`, `S`, and `xr` must be compatible with the minimal-energy configuration.

## Reference state

For optimal energy, `xr` can be one of the supported symbolic reference states:

In [ ]:
transitioner = Transitioner(T=1.0, energy_type="optimal")
energy = transitioner.fit_transform(A=A, X=X, xr="xf")
energy

Supported string values are:

- `"zero"`
- `"x0"`
- `"xf"`
- `"midpoint"`

A custom reference state can also be supplied as a supported empirical state input. The reference state must describe exactly one state.

## Labels and schema

`braincontrol` preserves labels when they can be inferred from pandas inputs.

For a `DataFrame`:

- rows represent states;
- the index provides state labels;
- columns provide node labels.

In [ ]:
import pandas as pd

states = pd.DataFrame(
    [
        [1.0, 0.0, 0.0],
        [0.0, 1.0, 0.0],
        [0.0, 0.0, 1.0],
    ],
    index=["rest", "task", "recovery"],
    columns=["node_A", "node_B", "node_C"],
)

adjacency = pd.DataFrame(
    A,
    index=states.columns,
    columns=states.columns,
)

transitioner = Transitioner(T=1.0, transitions="directed")
energy = transitioner.fit_transform(A=adjacency, X=states)
energy

Node labels form part of the fitted schema. Transform-time node labels must therefore be compatible with the labels learned during `fit`.

State labels are transform-time metadata rather than fitted schema. They are used to label the transitions produced by the current transform call.

A pandas `Series` represents a single state: its index provides node labels, and its `name`, when present, provides the state label.

Pandas `MultiIndex` objects are supported for hierarchical node and state metadata.

### Separate `X0` and `Xf` labels

When `X0` and `Xf` are supplied separately, they must either both provide state labels or both be unlabeled. If hierarchical state labels are represented by a `MultiIndex`, `X0` and `Xf` must use compatible MultiIndex structures.

## Neuroimaging input

The following example downloads a Schaefer atlas, a matching ENIGMA structural connectome, and statistical maps from NeuroVault. It requires an internet connection and may take some time on the first run.

The statistical maps are treated as states. A Nilearn masker converts each map to Schaefer parcel values, while the structural connectome provides the adjacency matrix.


In [ ]:
from nilearn.datasets import fetch_atlas_schaefer_2018
from nilearn.maskers import NiftiLabelsMasker
from braincontrol.datasets import (
    fetch_schaefer_structural_connectome,
    fetch_neurovault_stat_maps,
)

n_rois = 200

# Download adjacency matrix and statistical maps.
A_df = fetch_schaefer_structural_connectome(n_rois=n_rois)
df_stat_maps = fetch_neurovault_stat_maps()

# Download the matching atlas.
atlas = fetch_atlas_schaefer_2018(n_rois=n_rois)
atlas_img = atlas["maps"]
atlas_lut = atlas["lut"]

# Select three downloaded maps as states.
state_images = df_stat_maps["path"].dropna().tolist()[:3]
if len(state_images) < 2:
    raise ValueError("At least two NeuroVault maps are required.")

masker = NiftiLabelsMasker(
    labels_img=atlas_img,
    lut=atlas_lut,
    standardize=None,
    reports=False,
)

transitioner = Transitioner(
    T=1.0,
    transitions="directed",
    masker=masker,
)

# Use array values to avoid assuming that ENIGMA and Nilearn label
# strings have identical spelling; parcel ordering must still match.
energy = transitioner.fit_transform(
    A=A_df.to_numpy(),
    X=state_images,
)
energy


For each Niimg-like input, `Transitioner` clones the configured masker, fits it on that input, and transforms the image data into a node-level array. The user-provided masker instance itself is not fitted in place.

The resulting node representation must be compatible with the node schema established from the other empirical inputs.

## Storing trajectories

State and control trajectories can optionally be retained during `transform`:

In [ ]:
transitioner = Transitioner(
    T=1.0,
    transitions="directed",
    store_state_trajectories=True,
    store_control_trajectories=True,
)

energy = transitioner.fit_transform(A=A, X=X)

Retrieve them with:

In [ ]:
state_trajectories = transitioner.get_state_trajectories()
control_trajectories = transitioner.get_control_trajectories()
state_trajectories

The getters return `xarray.DataArray` objects with dimensions `("time", "node", "transition")`.

Available node and transition labels are included as coordinates. If the corresponding storage option is disabled, the getter returns `None`.

## Numerical errors

Numerical errors reported by `nctpy` for the most recent transform are available with:

In [ ]:
errors = transitioner.get_errors()
errors

## Main `Transitioner` parameters

```python
Transitioner(
    T,
    normalize_A=True,
    c=1,
    energy_type="optimal",
    rho=1.0,
    system="continuous",
    expm_version="scipy",
    transitions="directed",
    masker=None,
    memory=None,
    memory_level=1,
    verbose=0,
    store_state_trajectories=False,
    store_control_trajectories=False,
)
```

The empirical inputs `A`, `B`, `S`, `X`, `X0`, `Xf`, and `xr` are supplied to `fit`, `transform`, or `fit_transform`, rather than to the constructor.